In [1]:
import os
import cv2
import glob
import numpy as np
import joblib
from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [2]:
def extract_features(image_path):
    # 1. Load image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    # 2. Resize for consistency (optional but recommended for ML)
    img = cv2.resize(img, (256, 256))
    
    # 3. Apply Global Histogram Equalization (GHE)
    equalized_img = cv2.equalizeHist(img)
    
    # 4. Extract GLCM Textures
    glcm = graycomatrix(equalized_img, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]
    
    return [contrast, homogeneity, energy, correlation]

In [3]:
features = []
labels = []

# Define your paths (Change these to match your actual folders)
healthy_path = "dataset/healthy/*.jpg"
caries_path = "dataset/caries/*.jpg"

print("Processing Healthy X-rays...")
for img_path in glob.glob(healthy_path):
    feats = extract_features(img_path)
    if feats:
        features.append(feats)
        labels.append(0) # 0 = Healthy

print("Processing Caries X-rays...")
for img_path in glob.glob(caries_path):
    feats = extract_features(img_path)
    if feats:
        features.append(feats)
        labels.append(1) # 1 = Caries

X = np.array(features)
y = np.array(labels)
print(f"Dataset built! Total images: {len(X)}")

Processing Healthy X-rays...
Processing Caries X-rays...
Dataset built! Total images: 3211


In [4]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Save the trained model to a file
joblib.dump(model, 'caries_predictor.pkl')
print("Model saved as 'caries_predictor.pkl'")

Accuracy: 0.838258164852255

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.85      0.85       350
           1       0.82      0.82      0.82       293

    accuracy                           0.84       643
   macro avg       0.84      0.84      0.84       643
weighted avg       0.84      0.84      0.84       643

Model saved as 'caries_predictor.pkl'
